## Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. 
Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

We define a tool using @tool decorators before function definition.
We Link tools with model using bind_tools function and passing of tools to bind_tools using array ex - model.bind_tools([tool1, tool2])

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [5]:
model = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [7]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [8]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_2170826': 'EtgCCtUCAWkUfRMBCdgM6cLURw3IrEYpHaCUXgoQ/thfHo6xmY8F0t8wNZJxP6yZDGKjSNSH4NKzhnEyR9O+QcZaNaj7fYJkPzrAw0pxfhhLr2Ls0zyvFBL7rgyOma0Zqntio5AUNGaYoLAYQmOYelyTbj3acPzNjdj4dNDEvwMC7uWrCW62lYJ1XeLYVKoiRPg5ogkY0jDW7ZKaDN88BvJy1oGxFltHXIsSr4ag+0vY4k0CLegvFKWAQkVaMV5naiDa9k8zQ9ggWeaPTy9DEzmNPUOVXn60dMkfYBd/dNVPXEmHGi2lpchSVSDYUg+FriPA23ORqsVZXvGbf7hR04bt/KxyVLM2TwXcvmRvZu9N0G36053Hew5KtIVjT8/RVlINaoAunlJl+qaPIw9QoTU/vtbD7MepYNCbRwhruc2HuoXtf4Z1ljXDnarj4dy9HXIbMgEVkTmHFEM='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0c2b6-fe87-78f2-8b06-b215fbea75f0-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_2170826', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 

## Tool Execution loop

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is currently sunny.
